# FastAPI Exercises for AI Engineering

## Learning Goal

By the end of this notebook, you will understand how to use **FastAPI** to build API services for AI/ML systems.

You will learn:

- What FastAPI is
- How to create API endpoints
- How to run a FastAPI app with Uvicorn
- How request and response bodies work
- How to use Pydantic models for validation
- How FastAPI automatically creates API documentation
- How FastAPI appears in real AI engineering systems

### Background & Links

Useful official resources:

- FastAPI documentation: https://fastapi.tiangolo.com/
- FastAPI tutorial: https://fastapi.tiangolo.com/tutorial/
- Request body tutorial: https://fastapi.tiangolo.com/tutorial/body/
- Path operation advanced configuration: https://fastapi.tiangolo.com/advanced/path-operation-advanced-configuration/
- Pydantic documentation: https://docs.pydantic.dev/
- Uvicorn documentation: https://www.uvicorn.org/

### Why it matters in real AI engineering

FastAPI is commonly used to expose AI systems as services.

In real projects, your model, retriever, or pipeline usually needs an API so that other systems can call it.

Examples:

- A frontend calls `/ask` for a RAG answer
- A data pipeline calls `/ingest` to process files
- A monitoring system calls `/health`
- A Streamlit demo calls a FastAPI backend
- A production app calls `/embed`, `/search`, or `/generate`

FastAPI is important because it helps turn AI code into usable software.

## Mental Model

FastAPI is the **service layer** around your AI logic.

Your AI code might live in Python functions:

```python
retrieve_chunks(question)
generate_answer(question, chunks)
evaluate_answer(answer)
```

FastAPI exposes those functions through HTTP endpoints:

```text
POST /search
POST /ask
POST /evaluate
GET /health
```

### Big-picture explanation

```text
Client / frontend / script
        ↓ HTTP request
FastAPI endpoint
        ↓ validates input
AI service function
        ↓ returns result
FastAPI response
        ↓ JSON
Client receives output
```

### How this concept fits production systems

In production AI systems, FastAPI usually sits between:

- frontend apps
- Streamlit demos
- backend services
- vector databases
- embedding models
- LLM providers
- evaluation systems
- monitoring tools

FastAPI should not contain all your AI logic directly.

A better structure is:

```text
api/
  main.py
src/
  retriever.py
  generator.py
  schemas.py
  evaluation.py
  settings.py
```

The API receives requests and returns responses. The real AI logic should live in clean, testable modules.

## Background Explanation

### Core concepts

#### 1. App

A FastAPI app is created like this:

```python
from fastapi import FastAPI

app = FastAPI()
```

#### 2. Endpoint

An endpoint is a URL path plus an HTTP method.

Examples:

```python
@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/ask")
def ask(request: AskRequest):
    return {"answer": "..."}
```

#### 3. HTTP methods

Common methods:

- `GET`: read data
- `POST`: create or process data
- `PUT`: replace data
- `PATCH`: partially update data
- `DELETE`: delete data

For AI systems:

- `GET /health`
- `POST /ask`
- `POST /search`
- `POST /embed`
- `POST /ingest`
- `POST /evaluate`

#### 4. Request body

A request body is data sent by the client to the API.

Example JSON request:

```json
{
  "question": "What is RAG?",
  "top_k": 3
}
```

#### 5. Response body

A response body is data returned by the API.

Example JSON response:

```json
{
  "answer": "RAG retrieves relevant context before generation.",
  "sources": ["rag_intro.md"]
}
```

#### 6. Pydantic models

FastAPI uses Pydantic models to validate incoming and outgoing data.

Example:

```python
from pydantic import BaseModel

class AskRequest(BaseModel):
    question: str
    top_k: int = 3
```

#### 7. Automatic docs

FastAPI automatically creates interactive docs.

After running your app, open:

```text
http://127.0.0.1:8000/docs
```

You can test endpoints directly from the browser.

## Setup

FastAPI apps are normally run as Python scripts with Uvicorn.

In this notebook, you will:

1. Generate `.py` app files
2. Run them from your terminal
3. Open the docs page in your browser
4. Test endpoints interactively

Example command:

```bash
uvicorn fastapi_exercises.minimal_app:app --reload
```

Then open:

```text
http://127.0.0.1:8000/docs
```

In [4]:
# Install FastAPI and Uvicorn if needed.
# Run this cell once in a fresh environment.

import sys
import subprocess
import importlib.util

packages = {
    "fastapi": "fastapi",
    "uvicorn": "uvicorn",
    "pydantic": "pydantic",
}

missing = [pip_name for module_name, pip_name in packages.items() if importlib.util.find_spec(module_name) is None]

if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
else:
    print("FastAPI, Uvicorn, and Pydantic are already installed.")

FastAPI, Uvicorn, and Pydantic are already installed.


In [7]:
from pathlib import Path

APP_DIR = Path(".")
APP_DIR.mkdir(exist_ok=True)

init_file = APP_DIR / "__init__.py"
init_file.write_text("", encoding="utf-8")

print("App directory:", APP_DIR.resolve())

App directory: C:\AI\projects\lab\coding 101\daily\140526_fastAPI


## Minimal Working Example

This example creates a tiny FastAPI app with:

- a root endpoint
- a health endpoint
- a simple POST endpoint

After running the cell, start the app from your terminal:

```bash
uvicorn fastapi_exercises.minimal_app:app --reload
```

Then open:

```text
http://127.0.0.1:8000/docs
```

In [8]:
minimal_app_code = """
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI(
    title="Minimal FastAPI App",
    description="A tiny FastAPI app for AI engineering exercises.",
    version="0.1.0",
)

class EchoRequest(BaseModel):
    text: str

@app.get("/")
def read_root():
    return {"message": "Hello from FastAPI"}

@app.get("/health")
def health_check():
    return {"status": "ok"}

@app.post("/echo")
def echo(request: EchoRequest):
    return {
        "original_text": request.text,
        "uppercase_text": request.text.upper(),
        "length": len(request.text),
    }
"""

minimal_path = APP_DIR / "minimal_app.py"
minimal_path.write_text(minimal_app_code, encoding="utf-8")

print(f"Created: {minimal_path}")
print("Run this in your terminal:")
print("uvicorn fastapi_exercises.minimal_app:app --reload")

Created: minimal_app.py
Run this in your terminal:
uvicorn fastapi_exercises.minimal_app:app --reload


### Short explanation

This app has three endpoints:

```text
GET  /
GET  /health
POST /echo
```

The `/echo` endpoint receives a JSON body like:

```json
{
  "text": "hello ai engineer"
}
```

FastAPI validates that `text` exists and is a string.

This is the same idea used in AI APIs, except instead of echoing text, the endpoint might:

- embed text
- search a vector database
- call an LLM
- return an answer with citations

## Exercise 1

### Goal

Create a simple FastAPI app for a fake AI question-answering service.

Your app should include:

- `GET /health`
- `POST /ask`
- request model with:
  - `question: str`
  - `top_k: int = 3`
- response containing:
  - question
  - fake answer
  - top_k
  - fake sources

### TODO section

Complete the generated file:

```text
fastapi_exercises/exercise_1_todo.py
```

Run it with:

```bash
uvicorn fastapi_exercises.exercise_1_todo:app --reload
```

### Questions to think about

- Why should an AI API validate `question` and `top_k`?
- Why is `POST /ask` better than `GET /ask` for complex AI requests?
- Why should an answer endpoint return sources?

In [9]:
exercise_1_todo = """
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI(title="Exercise 1 AI QA API")

# TODO 1:
# Create a Pydantic model called AskRequest with:
# - question: str
# - top_k: int = 3

# TODO 2:
# Create GET /health endpoint that returns {"status": "ok"}

# TODO 3:
# Create POST /ask endpoint.
# It should accept AskRequest and return:
# - question
# - answer
# - top_k
# - sources

# Example fake answer:
# "This is a fake AI answer. In production, this would come from a RAG pipeline."
"""

exercise_1_path = APP_DIR / "exercise_1_todo.py"
exercise_1_path.write_text(exercise_1_todo, encoding="utf-8")

print(f"Created: {exercise_1_path}")
print("Run after completing:")
print("uvicorn fastapi_exercises.exercise_1_todo:app --reload")

Created: exercise_1_todo.py
Run after completing:
uvicorn fastapi_exercises.exercise_1_todo:app --reload


## Solution 1

In [ ]:
solution_1_code = """
from fastapi import FastAPI
from pydantic import BaseModel, Field

app = FastAPI(
    title="AI QA API",
    description="A simple fake question-answering API.",
    version="0.1.0",
)

class AskRequest(BaseModel):
    question: str = Field(..., min_length=3)
    top_k: int = Field(3, ge=1, le=10)

@app.get("/health")
def health_check():
    return {"status": "ok"}

@app.post("/ask")
def ask(request: AskRequest):
    return {
        "question": request.question,
        "answer": (
            "This is a fake AI answer. In production, this would come from "
            "a retriever plus an LLM generator."
        ),
        "top_k": request.top_k,
        "sources": [
            {"source": "rag_intro.md", "score": 0.91},
            {"source": "vector_db.md", "score": 0.86},
        ][: request.top_k],
    }
"""

solution_1_path = APP_DIR / "solution_1_ai_qa_api.py"
solution_1_path.write_text(solution_1_code, encoding="utf-8")

print(f"Created: {solution_1_path}")
print("Run:")
print("uvicorn fastapi_exercises.solution_1_ai_qa_api:app --reload")

## Exercise 2

### Slightly harder

Create a mini retrieval API.

This app should simulate a retriever endpoint like the one used in RAG systems.

### Goal

Build an API with:

- `POST /search`
- request model:
  - `query: str`
  - `top_k: int = 3`
  - `source_filter: str | None = None`
- response:
  - query
  - top_k
  - results list

Each result should contain:

- text
- source
- score

### Questions to think about

- Why do search APIs usually include `top_k`?
- Why is a `source_filter` useful?
- Where would Qdrant be called in a real version of this endpoint?
- Should retrieval results include scores? Why?

In [10]:
exercise_2_todo = """
from typing import Optional
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI(title="Exercise 2 Retrieval API")

FAKE_CHUNKS = [
    {
        "text": "RAG retrieves relevant chunks before generation.",
        "source": "rag_notes.md",
        "score": 0.92,
    },
    {
        "text": "Vector databases store embeddings and metadata payloads.",
        "source": "qdrant_notes.md",
        "score": 0.88,
    },
    {
        "text": "FastAPI exposes AI functionality through HTTP endpoints.",
        "source": "fastapi_notes.md",
        "score": 0.84,
    },
]

# TODO 1:
# Create SearchRequest model with:
# - query: str
# - top_k: int = 3
# - source_filter: Optional[str] = None

# TODO 2:
# Create POST /search endpoint.
# It should:
# - filter FAKE_CHUNKS by source if source_filter is provided
# - return only top_k results
# - include query and top_k in response

# TODO 3:
# Add validation with Field:
# - query minimum length 3
# - top_k between 1 and 10
"""

exercise_2_path = APP_DIR / "exercise_2_todo.py"
exercise_2_path.write_text(exercise_2_todo, encoding="utf-8")

print(f"Created: {exercise_2_path}")
print("Run after completing:")
print("uvicorn fastapi_exercises.exercise_2_todo:app --reload")

Created: exercise_2_todo.py
Run after completing:
uvicorn fastapi_exercises.exercise_2_todo:app --reload


In [ ]:
FAKE_CHUNKS = [
    {
        "text": "RAG retrieves relevant chunks before generation.",
        "source": "rag_notes.md",
        "score": 0.92,
    },
    {
        "text": "Vector databases store embeddings and metadata payloads.",
        "source": "qdrant_notes.md",
        "score": 0.88,
    },
    {
        "text": "FastAPIb exposes AI functionality through HTTP endpoints.",
        "source": "fastapi_notes.md",
        "score": 0.84,
    },
]

filter = [x for x in FAKE_CHUNKS if (x["source"] == "fastapi_notes.md")]

print(filter)

# if not None:
#     print("123")

[{'text': 'FastAPI exposes AI functionality through HTTP endpoints.', 'source': 'fastapi_notes.md', 'score': 0.84}]


## Solution 2

In [ ]:
solution_2_code = """
from typing import Optional
from fastapi import FastAPI
from pydantic import BaseModel, Field

app = FastAPI(
    title="Mini Retrieval API",
    description="A fake retrieval API similar to what a RAG backend would expose.",
    version="0.1.0",
)

FAKE_CHUNKS = [
    {
        "text": "RAG retrieves relevant chunks before generation.",
        "source": "rag_notes.md",
        "score": 0.92,
    },
    {
        "text": "Vector databases store embeddings and metadata payloads.",
        "source": "qdrant_notes.md",
        "score": 0.88,
    },
    {
        "text": "FastAPI exposes AI functionality through HTTP endpoints.",
        "source": "fastapi_notes.md",
        "score": 0.84,
    },
    {
        "text": "Evaluation checks whether retrieved chunks support the answer.",
        "source": "evaluation_notes.md",
        "score": 0.80,
    },
]

class SearchRequest(BaseModel):
    query: str = Field(..., min_length=3)
    top_k: int = Field(3, ge=1, le=10)
    source_filter: Optional[str] = None

@app.get("/health")
def health_check():
    return {"status": "ok"}

@app.post("/search")
def search(request: SearchRequest):
    results = FAKE_CHUNKS

    if request.source_filter:
        results = [
            chunk for chunk in results
            if chunk["source"] == request.source_filter
        ]

    results = results[: request.top_k]

    return {
        "query": request.query,
        "top_k": request.top_k,
        "source_filter": request.source_filter,
        "results": results,
    }
"""

solution_2_path = APP_DIR / "solution_2_retrieval_api.py"
solution_2_path.write_text(solution_2_code, encoding="utf-8")

print(f"Created: {solution_2_path}")
print("Run:")
print("uvicorn fastapi_exercises.solution_2_retrieval_api:app --reload")

## Common Mistakes

### Mistake 1: Running the file with plain Python

Wrong:

```bash
python main.py
```

Better:

```bash
uvicorn main:app --reload
```

If your file is inside a package:

```bash
uvicorn fastapi_exercises.minimal_app:app --reload
```

### Mistake 2: Forgetting `:app`

Wrong:

```bash
uvicorn fastapi_exercises.minimal_app --reload
```

Correct:

```bash
uvicorn fastapi_exercises.minimal_app:app --reload
```

`app` is the FastAPI object inside the Python file.

### Mistake 3: Not using Pydantic request models

Bad:

```python
@app.post("/ask")
def ask(question: str, top_k: int):
    ...
```

This expects query parameters, not a clean JSON body.

Better:

```python
class AskRequest(BaseModel):
    question: str
    top_k: int = 3

@app.post("/ask")
def ask(request: AskRequest):
    ...
```

### Mistake 4: Putting model loading inside every request

Bad:

```python
@app.post("/ask")
def ask(request: AskRequest):
    model = load_large_model()
    return model.generate(request.question)
```

Better:

```python
model = load_large_model()

@app.post("/ask")
def ask(request: AskRequest):
    return model.generate(request.question)
```

For bigger systems, use startup/lifespan patterns or dependency injection.

### Mistake 5: Mixing API and business logic too much

Bad:

```text
main.py has endpoints, chunking, embedding, retrieval, generation, evaluation, and database code.
```

Better:

```text
api/main.py
src/retriever.py
src/generator.py
src/embedder.py
src/evaluator.py
```

Keep the API thin and the AI logic modular.

### Mistake 6: Returning unstructured responses

Bad:

```python
return "answer text"
```

Better:

```python
return {
    "answer": answer,
    "sources": sources,
    "metadata": metadata,
}
```

Structured responses are easier for frontends, tests, and monitoring.

In [ ]:
debugging_app_code = """
from fastapi import FastAPI
from pydantic import BaseModel, Field

app = FastAPI(title="FastAPI Debugging Example")

class TopKRequest(BaseModel):
    query: str = Field(..., min_length=3)
    top_k: int = Field(3, ge=1, le=5)

@app.post("/debug-search")
def debug_search(request: TopKRequest):
    return {
        "message": "Validation passed.",
        "query": request.query,
        "top_k": request.top_k,
    }
"""

debugging_path = APP_DIR / "debugging_validation_example.py"
debugging_path.write_text(debugging_app_code, encoding="utf-8")

print(f"Created: {debugging_path}")
print("Run:")
print("uvicorn fastapi_exercises.debugging_validation_example:app --reload")
print()
print("Then test invalid top_k values in /docs, such as top_k = 99.")

## Real AI Engineering Usage

### 1. RAG

FastAPI often exposes RAG endpoints.

Example:

```text
POST /ask
POST /search
POST /ingest
GET  /health
```

Typical RAG request:

```json
{
  "question": "What does this book say about chunking?",
  "top_k": 5,
  "filters": {
    "book": "Arabic NLP Guide"
  }
}
```

Typical RAG response:

```json
{
  "answer": "...",
  "sources": [
    {
      "text": "...",
      "book": "Arabic NLP Guide",
      "page": 12,
      "score": 0.91
    }
  ]
}
```

### 2. APIs

FastAPI is the API layer itself.

It provides:

- request validation
- response serialization
- HTTP routes
- automatic docs
- dependency injection
- error handling
- middleware support

### 3. ML pipelines

FastAPI can serve:

- classification models
- embedding models
- rerankers
- text generators
- OCR services
- feature extraction pipelines

Example:

```text
POST /predict
POST /embed
POST /classify
POST /rerank
```

### 4. Ingestion

An ingestion API might expose:

```text
POST /documents/upload
POST /documents/ingest
GET  /documents/{document_id}/status
```

The endpoint may start processing files, extracting text, chunking content, and storing embeddings.

### 5. Evaluation

A FastAPI evaluation service might expose:

```text
POST /evaluate/retrieval
POST /evaluate/answer
POST /evaluate/batch
```

This is useful when evaluation becomes part of CI, dashboards, or internal QA tooling.

### 6. Vector DBs

FastAPI often wraps vector DB operations.

Example flow:

```text
POST /search
    ↓
embed query
    ↓
Qdrant search
    ↓
return top-k points
```

FastAPI should not replace Qdrant. It should expose a controlled interface around Qdrant.

## Final Mini Exercise

### Small production-style task

Create a FastAPI app called:

```text
fastapi_exercises/final_rag_api.py
```

The app should simulate a small RAG backend.

It should include:

1. `GET /health`
2. `POST /search`
3. `POST /ask`
4. Pydantic request models
5. input validation
6. fake retrieved chunks
7. fake generated answer
8. structured JSON responses

This is similar to the first backend API you might build for a RAG demo.

In [ ]:
final_todo_code = """
from fastapi import FastAPI
from pydantic import BaseModel, Field

app = FastAPI(title="Final Mini RAG API")

FAKE_CHUNKS = [
    {
        "text": "RAG retrieves relevant chunks before generation.",
        "source": "rag_intro.md",
        "score": 0.92,
    },
    {
        "text": "Vector databases store embeddings for semantic search.",
        "source": "vector_db.md",
        "score": 0.88,
    },
    {
        "text": "FastAPI exposes AI systems through HTTP endpoints.",
        "source": "fastapi.md",
        "score": 0.84,
    },
]

# TODO:
# 1. Create SearchRequest model:
#    - query: str, min length 3
#    - top_k: int, default 3, between 1 and 10
#
# 2. Create AskRequest model:
#    - question: str, min length 3
#    - top_k: int, default 3, between 1 and 10
#
# 3. Create GET /health
#
# 4. Create POST /search
#    - return query, top_k, and fake top-k chunks
#
# 5. Create POST /ask
#    - internally reuse fake top-k chunks
#    - return question, answer, and sources
"""

final_todo_path = APP_DIR / "final_rag_api_todo.py"
final_todo_path.write_text(final_todo_code, encoding="utf-8")

print(f"Created: {final_todo_path}")
print("Run after completing:")
print("uvicorn fastapi_exercises.final_rag_api_todo:app --reload")

### Final Mini Exercise Solution

In [ ]:
final_solution_code = """
from fastapi import FastAPI
from pydantic import BaseModel, Field

app = FastAPI(
    title="Final Mini RAG API",
    description="A small production-style mock RAG backend.",
    version="0.1.0",
)

FAKE_CHUNKS = [
    {
        "text": "RAG retrieves relevant chunks before generation.",
        "source": "rag_intro.md",
        "score": 0.92,
    },
    {
        "text": "Vector databases store embeddings for semantic search.",
        "source": "vector_db.md",
        "score": 0.88,
    },
    {
        "text": "FastAPI exposes AI systems through HTTP endpoints.",
        "source": "fastapi.md",
        "score": 0.84,
    },
    {
        "text": "Evaluation checks if retrieved context supports the answer.",
        "source": "evaluation.md",
        "score": 0.80,
    },
]

class SearchRequest(BaseModel):
    query: str = Field(..., min_length=3)
    top_k: int = Field(3, ge=1, le=10)

class AskRequest(BaseModel):
    question: str = Field(..., min_length=3)
    top_k: int = Field(3, ge=1, le=10)

@app.get("/health")
def health_check():
    return {
        "status": "ok",
        "service": "final-mini-rag-api",
    }

@app.post("/search")
def search(request: SearchRequest):
    results = FAKE_CHUNKS[: request.top_k]

    return {
        "query": request.query,
        "top_k": request.top_k,
        "results": results,
    }

@app.post("/ask")
def ask(request: AskRequest):
    sources = FAKE_CHUNKS[: request.top_k]

    answer = (
        "A RAG system embeds the user question, retrieves relevant chunks from a "
        "vector database, and sends those chunks to an LLM so the answer is grounded "
        "in source material."
    )

    return {
        "question": request.question,
        "answer": answer,
        "sources": sources,
        "metadata": {
            "retriever": "fake-retriever",
            "generator": "fake-generator",
            "top_k": request.top_k,
        },
    }
"""

final_solution_path = APP_DIR / "final_rag_api_solution.py"
final_solution_path.write_text(final_solution_code, encoding="utf-8")

print(f"Created: {final_solution_path}")
print("Run:")
print("uvicorn fastapi_exercises.final_rag_api_solution:app --reload")

## Key Takeaways

- FastAPI is a Python framework for building HTTP APIs.
- It is commonly used as the service layer for AI systems.
- Use `uvicorn module_name:app --reload` to run a development server.
- `GET` is usually for reading data; `POST` is usually for sending structured requests.
- Pydantic models validate request bodies and make APIs safer.
- FastAPI automatically generates interactive API docs at `/docs`.
- AI endpoints should return structured JSON, not plain strings.
- Keep API code thin and move AI logic into separate modules.
- In RAG, FastAPI often exposes `/search`, `/ask`, `/ingest`, and `/evaluate`.
- FastAPI is often used together with vector databases, LLMs, embedding models, and frontend apps.